In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

from src.data import (
    load_sales, load_web_traffic, 
    load_promotions, load_order_items, 
    load_orders
)
from src.features import add_time_features, add_lag_features, list_feature_columns, one_hot_encode
from src.pipelines.infer import fit_and_predict
from src.models import SklearnRegressorConfig, SklearnRegressorWrapper
from src.utils import prepare_submission
import pandas as pd


DATA_ROOT = project_root / "data/datathon-2026-round-1"

In [ ]:
TRAIN_RANGE = ("2013-01-01", "2022-12-31")
PREDICT_RANGE = ("2023-01-01", "2024-07-01")
df_sales = load_sales(data_root=DATA_ROOT)
df_traffic = load_web_traffic(data_root=DATA_ROOT)

df_traffic_test = pd.read_csv(f"{project_root}/results/webtraffic_predictions.csv", parse_dates=["date"])
df_traffic_test.set_index("date", inplace=True)

In [ ]:
TRAFFIC_FEATURES_RAW = ["date", "sessions", "unique_visitors"]
SALE_FEATURES_RAW = ["date", "Revenue", "COGS"]

TRAFFIC_ENGINEER = ["date", "returning_rate"]

In [ ]:
df = pd.DataFrame(
    {"date": pd.date_range(start=TRAIN_RANGE[0], end=PREDICT_RANGE[1], freq="D")}
)
df = df.merge(df_sales[SALE_FEATURES_RAW], on="date", how="left")
df = df.merge(df_traffic[TRAFFIC_FEATURES_RAW], on="date", how="left")
df.set_index("date", inplace=True)
df.update(df_traffic_test[TRAFFIC_FEATURES_RAW[1:]])
df.reset_index(inplace=True)

df["returning_rate"] = (df["sessions"] - df["unique_visitors"])/ df["sessions"]
df.drop(columns=["sessions", "unique_visitors"], inplace=True)

In [ ]:
df = add_time_features(df, date_col="date")
categorical_cols = ["date_of_week", "month", "day", "is_weekend"]

df = one_hot_encode(df, categorical_cols)

In [ ]:
model_config = SklearnRegressorConfig(
    model_type="lightgbm",
)

submission = fit_and_predict(
    df=df,
    model_config=model_config,
    train_range=TRAIN_RANGE,
    predict_range=PREDICT_RANGE,
)
submission.head()

In [ ]:
submission.to_csv(project_root / "submissions/lightgbm_2.csv", index=False)